In [1]:

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — switch to GPU runtime!")
assert torch.cuda.is_available(), "Please enable GPU: Runtime → Change runtime type → T4 GPU"

# Install all dependencies
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("langchain", "langchain-community", "langchain-huggingface")
pip("faiss-cpu", "sentence-transformers", "pypdf")
pip("transformers", "accelerate", "bitsandbytes")
pip("openai-whisper", "ffmpeg-python")
pip("fastapi", "uvicorn[standard]", "python-multipart", "aiofiles")
pip("pyngrok", "nest-asyncio")
pip("gtts")                  # text-to-speech
pip("huggingface_hub")

# Install system ffmpeg (needed by Whisper)
subprocess.check_call(["apt-get", "install", "-qq", "-y", "ffmpeg"])

print("\nAll dependencies installed.")


CUDA available: True
GPU: Tesla T4

All dependencies installed.


In [2]:
!pip uninstall -y langchain langchain-community langchain-core langchain-huggingface

Found existing installation: langchain 1.2.12
Uninstalling langchain-1.2.12:
  Successfully uninstalled langchain-1.2.12
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: langchain-core 1.2.19
Uninstalling langchain-core-1.2.19:
  Successfully uninstalled langchain-core-1.2.19
Found existing installation: langchain-huggingface 1.2.1
Uninstalling langchain-huggingface-1.2.1:
  Successfully uninstalled langchain-huggingface-1.2.1


In [3]:
!pip install \
langchain==0.1.16 \
langchain-community==0.0.34 \
faiss-cpu \
pypdf \
sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.4
    Uninstalling tenacity-9.1.4:
      Successfully uninstalled tenacity-9.1.4
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfu

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# 🔁 Change this to your folder name
%cd /content/drive/MyDrive/ColabNotebooks/as-sunnah

import os
print("Current files:", os.listdir())

Mounted at /content/drive
/content/drive/MyDrive/ColabNotebooks/as-sunnah
Current files: ['app_state.py', 'env', 'tts_output', 'docs', '__pycache__', 'main.py', 'voice_pipeline.py', 'index.html', 'faiss_index', 'rag_pipeline.py', 'colab_setup.ipynb']


In [2]:
!pip install python-dotenv -q

from dotenv import load_dotenv
import os

load_dotenv("env")

hf_token = os.getenv("HF_TOKEN")
ngrok_token = os.getenv("NGROK_TOKEN")

print("HF:", bool(hf_token))
print("NGROK:", bool(ngrok_token))

HF: True
NGROK: True


In [3]:
from huggingface_hub import login
from pyngrok import conf

login(hf_token)
conf.get_default().auth_token = ngrok_token

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
DOCS_DIR = "./docs"

import os
print("Docs found:", os.listdir(DOCS_DIR))

Docs found: ['zakat_distribution.pdf', 'zakat_calculation.pdf', 'zakat_education.pdf', 'zakat_faq.pdf', 'zakat_policy.pdf']


In [5]:
from pathlib import Path
from rag_pipeline import build_index_from_docs, load_vectorstore

INDEX_PATH = "./faiss_index"

if Path(INDEX_PATH).exists():
    print("Loading existing index...")
    vectorstore = load_vectorstore(INDEX_PATH)
else:
    print("Building index from Drive PDFs...")
    vectorstore = build_index_from_docs(DOCS_DIR, index_path=INDEX_PATH)

print("Vectorstore ready.")

Loading existing index...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading FAISS index from: ./faiss_index/
Vectorstore ready.


In [6]:
from voice_pipeline import load_whisper

whisper_model = load_whisper(model_size="base")
print("Whisper ready.")

Loading Whisper 'base' on cuda...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 126MiB/s]


Whisper ready.
Whisper ready.


In [7]:
from rag_pipeline import load_llm

llm_pipeline = load_llm()
print("LLM ready.")


Loading LLM: TinyLlama/TinyLlama-1.1B-Chat-v1.0


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded.
LLM ready.


In [8]:
# from rag_pipeline import generate_answer

# query = "What is zakat?"
# result = generate_answer(query, vectorstore, llm_pipeline)

# print(result["answer"])

In [9]:
import nest_asyncio
import uvicorn
import threading
import time
from pyngrok import ngrok
import app_state

nest_asyncio.apply()

ngrok.kill()

# ✅ Inject shared state
app_state.vectorstore   = vectorstore
app_state.llm_pipeline  = llm_pipeline
app_state.whisper_model = whisper_model

PORT = 8000

def run():
    uvicorn.run("main:app", host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=run, daemon=True).start()

time.sleep(2)

tunnel = ngrok.connect(PORT)
print("🌐 Public URL:", tunnel.public_url)

🌐 Public URL: https://unplastic-kimiko-nonpermeable.ngrok-free.dev
